In [13]:
import math
import random
import yaml
import argparse
from dotmap import DotMap

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam

# import matplotlib.pyplot as plt
import wandb

import sys
sys.path.append("../src")  # make sure Python can find src/
import data
from model import GPTLinear, GPTSoftmax
from multi_task_train import train_step


def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    ## Not sure if below would work if I dont have gpu
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed set to {seed}")


def load_config(config_path: str):
    """Load YAML config and convert to DotMap."""
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    cfg = DotMap(cfg)
    return cfg


def prepare_data_samplers(config, device):
    """Create a dict of data samplers for each task."""
    num_task = len(config.data.tasks)
    data_samplers = {}
    for task_config in config.data.tasks:
        task_name = task_config.name
        task_class = getattr(data, task_name)
        data_samplers[task_name] = {
            "sampler": task_class(task_config, device),
            "n_train": task_config.n_train,
            "n_test": task_config.n_test,
        }
    return data_samplers

In [14]:
config = load_config('../src/configs/mix1_mws_mwp.yaml')

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [16]:
seed = getattr(config.train, "seed", 42)
set_seed(seed)

Seed set to 67


In [17]:
n_tasks = len(config.data.tasks)
config.model.vocab_size = max(getattr(config.data, "p", 17), config.data.max_num) + n_tasks
config.model.block_size = 2 * config.data.num_tokens + 1

In [32]:
data_samplers = prepare_data_samplers(config, device)
data_sampler_mws = {'MovingWindowSum': data_samplers["MovingWindowSum"]}
data_sampler_mwp = {'MovingWindowProduct': data_samplers["MovingWindowProduct"]}
data_sampler_mwp

{'MovingWindowProduct': {'sampler': <data.moving_window_product.MovingWindowProduct at 0x7f6be44da100>,
  'n_train': 64,
  'n_test': 32}}

In [33]:
if config.model.linear:
        model = GPTLinear(config.model, return_att=True).to(device)
else:
    model = GPTSoftmax(config.model, return_att=True).to(device)


In [34]:
if config.train.freeze_embedding: ### NEed to double check
        for param in model.transformer.wte.parameters():
            param.requires_grad = False
        for param in model.transformer.wpe.parameters():
            param.requires_grad = False

In [36]:
stop_on_perfect = getattr(config.train, "stop_on_perfect_acc", True)
stop_on_perfect = True

In [37]:
optim = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config.train.lr)

/home/jyue/.conda/envs/emerge/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if getattr(config.train, "wandb", False):
    wandb_run_name = "MWS_to_MWP_transfer"
    wandb.login(key="")
    wandb.init(project=config.train.wandb_project, name=wandb_run_name, config=config, save_code=False)
    if getattr(config.train, "wandb_save_model", False):
        wandb.watch(model, log="parameters")  # save model parameters
    else:   
        wandb.watch(model, log=None)        # metrics only, no model saved

stop_on_perfect = True
perfect_patience = 50
# acc_eps = getattr(config.train, "perfect_acc_eps", 1e-6)

perfect_counter = 0

# Training loop
global_step = 0
for step in range(config.train.num_steps):
    overall_metrics = train_step(
        model=model,
        optim=optim,
        data_samplers=data_sampler_mws,
        step=global_step,
        config=config,
        device=device,
    )
    global_step += 1
    ## Early stop
    if stop_on_perfect:
        acc = overall_metrics["test_acc"]  # or "train_acc" if you prefer

        if acc >= 1.0:
            perfect_counter += 1
        else:
            perfect_counter = 0

        if perfect_counter >= perfect_patience:
            print(
                f"\n✅ Early stopping at step {step}: "
                f"overall accuracy stayed at 100% for "
                f"{perfect_patience} consecutive steps\n"
            )
            break

# if getattr(config.train, "wandb", False):
#     wandb.finish()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: jetyue04 (wth_ucsd). Use `wandb login --relogin` to force relogin


Step 0 -- Train loss: 2.905679941177368, Train Acc: 0.05859375 Test Acc: 0.044921875
Step 1 -- Train loss: 2.906642436981201, Train Acc: 0.0625 Test Acc: 0.056640625
Step 2 -- Train loss: 2.9275994300842285, Train Acc: 0.0615234375 Test Acc: 0.052734375
Step 3 -- Train loss: 2.909619092941284, Train Acc: 0.0712890625 Test Acc: 0.05859375
Step 4 -- Train loss: 2.8919270038604736, Train Acc: 0.06640625 Test Acc: 0.068359375
Step 5 -- Train loss: 2.8863003253936768, Train Acc: 0.048828125 Test Acc: 0.076171875
Step 6 -- Train loss: 2.8626654148101807, Train Acc: 0.0673828125 Test Acc: 0.08203125
Step 7 -- Train loss: 2.8755507469177246, Train Acc: 0.0595703125 Test Acc: 0.0859375
Step 8 -- Train loss: 2.860372543334961, Train Acc: 0.080078125 Test Acc: 0.08203125
Step 9 -- Train loss: 2.8653852939605713, Train Acc: 0.0517578125 Test Acc: 0.056640625
Step 10 -- Train loss: 2.8574249744415283, Train Acc: 0.0634765625 Test Acc: 0.052734375
Step 11 -- Train loss: 2.8466105461120605, Train Acc

wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.3 seconds.), retrying request


Step 16 -- Train loss: 2.8354239463806152, Train Acc: 0.0732421875 Test Acc: 0.076171875
Step 17 -- Train loss: 2.8339526653289795, Train Acc: 0.060546875 Test Acc: 0.07421875
Step 18 -- Train loss: 2.831103801727295, Train Acc: 0.0830078125 Test Acc: 0.0703125
Step 19 -- Train loss: 2.8327953815460205, Train Acc: 0.08984375 Test Acc: 0.080078125
Step 20 -- Train loss: 2.8420536518096924, Train Acc: 0.0712890625 Test Acc: 0.068359375
Step 21 -- Train loss: 2.8238186836242676, Train Acc: 0.0810546875 Test Acc: 0.076171875
Step 22 -- Train loss: 2.814864158630371, Train Acc: 0.08984375 Test Acc: 0.095703125
Step 23 -- Train loss: 2.822632074356079, Train Acc: 0.08203125 Test Acc: 0.087890625


wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.2 seconds.), retrying request
wandb: 429 encountered (Filestream rate limit exceeded, retrying in 4.4 seconds.), retrying request


Step 24 -- Train loss: 2.8220653533935547, Train Acc: 0.08203125 Test Acc: 0.10546875
Step 25 -- Train loss: 2.8073527812957764, Train Acc: 0.0986328125 Test Acc: 0.099609375
Step 26 -- Train loss: 2.812925338745117, Train Acc: 0.0986328125 Test Acc: 0.072265625
Step 27 -- Train loss: 2.806082010269165, Train Acc: 0.0986328125 Test Acc: 0.11328125
Step 28 -- Train loss: 2.7969658374786377, Train Acc: 0.1162109375 Test Acc: 0.11328125
Step 29 -- Train loss: 2.784817934036255, Train Acc: 0.1025390625 Test Acc: 0.103515625
Step 30 -- Train loss: 2.780487060546875, Train Acc: 0.115234375 Test Acc: 0.12109375
Step 31 -- Train loss: 2.778848171234131, Train Acc: 0.1142578125 Test Acc: 0.12109375
Step 32 -- Train loss: 2.7724950313568115, Train Acc: 0.125 Test Acc: 0.138671875


wandb: 429 encountered (Filestream rate limit exceeded, retrying in 2.1 seconds.), retrying request


Step 33 -- Train loss: 2.763015031814575, Train Acc: 0.1171875 Test Acc: 0.115234375
Step 34 -- Train loss: 2.754838228225708, Train Acc: 0.1181640625 Test Acc: 0.109375
Step 35 -- Train loss: 2.7565338611602783, Train Acc: 0.1220703125 Test Acc: 0.126953125
Step 36 -- Train loss: 2.745964288711548, Train Acc: 0.125 Test Acc: 0.138671875
Step 37 -- Train loss: 2.7336761951446533, Train Acc: 0.119140625 Test Acc: 0.12109375
Step 38 -- Train loss: 2.733745574951172, Train Acc: 0.1181640625 Test Acc: 0.130859375
Step 39 -- Train loss: 2.7279865741729736, Train Acc: 0.1357421875 Test Acc: 0.115234375
Step 40 -- Train loss: 2.7230429649353027, Train Acc: 0.12109375 Test Acc: 0.115234375
Step 41 -- Train loss: 2.7162394523620605, Train Acc: 0.12109375 Test Acc: 0.12109375
Step 42 -- Train loss: 2.705120801925659, Train Acc: 0.1181640625 Test Acc: 0.130859375
Step 43 -- Train loss: 2.7045469284057617, Train Acc: 0.1162109375 Test Acc: 0.10546875
Step 44 -- Train loss: 2.696791172027588, Train

In [39]:
import torch

# After training loop finishes (or inside early stop block)
save_path = "mws_to_mwp/ckpt_after_MWS.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optim.state_dict(),
    "config": config,
    "step": step,
}, save_path)

print(f"Model saved to {save_path}")

Model saved to mws_to_mwp/ckpt_after_MWS.pt


In [40]:

perfect_counter = 0
global_step = 254
# Training loop
for step in range(config.train.num_steps):
    overall_metrics = train_step(
        model=model,
        optim=optim,
        data_samplers=data_sampler_mwp,
        step=global_step,
        config=config,
        device=device,
    )
    global_step +=1
    ## Early stop
    if stop_on_perfect:
        acc = overall_metrics["test_acc"]  # or "train_acc" if you prefer

        if acc >= 1.0:
            perfect_counter += 1
        else:
            perfect_counter = 0

        if perfect_counter >= perfect_patience:
            print(
                f"\n✅ Early stopping at step {step}: "
                f"overall accuracy stayed at 100% for "
                f"{perfect_patience} consecutive steps\n"
            )
            break

Step 254 -- Train loss: 4.929762840270996, Train Acc: 0.119140625 Test Acc: 0.1328125
Step 255 -- Train loss: 4.495541095733643, Train Acc: 0.1171875 Test Acc: 0.11328125
Step 256 -- Train loss: 3.7134406566619873, Train Acc: 0.126953125 Test Acc: 0.12109375
Step 257 -- Train loss: 3.358062744140625, Train Acc: 0.1044921875 Test Acc: 0.1015625
Step 258 -- Train loss: 3.105642318725586, Train Acc: 0.123046875 Test Acc: 0.1171875
Step 259 -- Train loss: 2.9988887310028076, Train Acc: 0.130859375 Test Acc: 0.119140625
Step 260 -- Train loss: 2.915714979171753, Train Acc: 0.1455078125 Test Acc: 0.13671875
Step 261 -- Train loss: 2.872286081314087, Train Acc: 0.134765625 Test Acc: 0.126953125
Step 262 -- Train loss: 2.7595374584198, Train Acc: 0.1591796875 Test Acc: 0.162109375
Step 263 -- Train loss: 2.720515251159668, Train Acc: 0.1513671875 Test Acc: 0.14453125
Step 264 -- Train loss: 2.6787314414978027, Train Acc: 0.1689453125 Test Acc: 0.171875
Step 265 -- Train loss: 2.665471553802490

In [41]:
if getattr(config.train, "wandb", False):
    wandb.finish()

wandb: WARNING Source type is set to 'repo' but some required information is missing from the environment. A job will not be created from this run. See https://docs.wandb.ai/guides/launch/create-job


MovingWindowProduct/att_prog_measure,▇▄▂▁▁▁▁▁▁▁▁▂▂▂▂▃▃▄▄▅▅▆▆▆▇▇▇▇▇▇██████████
MovingWindowProduct/examples_seen,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
MovingWindowProduct/idx0_check,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
MovingWindowProduct/idx10_check,▁▁▂▂▂▂▂▂▂▂▃▂▃▃▄▄▅▅▆▆▇▇▇█████████████████
MovingWindowProduct/idx11_check,▁▁▁▂▂▁▁▂▁▂▂▂▂▃▃▃▃▄▅▅▅▇▇▇████████████████
MovingWindowProduct/idx12_check,▁▁▁▂▁▁▁▁▁▁▁▁▂▁▂▃▂▄▄▅▆▇▇█████████████████
MovingWindowProduct/idx13_check,▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▄▅▄▅▅▇▇▇█████████████████
MovingWindowProduct/idx14_check,▁▁▁▁▂▂▁▁▂▂▂▂▂▃▃▄▅▅▆▅▇▇██████████████████
MovingWindowProduct/idx15_check,▁▁▂▁▂▁▁▁▁▂▂▂▂▁▂▂▄▅▆▆▆▇▇█████████████████
MovingWindowProduct/idx1_check,▁▁▁▂▂▁▂▂▂▂▃▃▃▃▃▄▅▆▅▆▆▇▇█████████████████
MovingWindowProduct/idx2_check,▁▁▁▂▂▂▁▂▂▂▂▃▃▄▃▄▄▅▅▆▆▇▇█████████████████


In [42]:
import torch

# After training loop finishes (or inside early stop block)
save_path = "mws_to_mwp/ckpt_after_MWP.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optim.state_dict(),
    "config": config,
    "step": step,
}, save_path)

print(f"Model saved to {save_path}")

Model saved to mws_to_mwp/ckpt_after_MWP.pt


In [45]:
import torch
import argparse
import math

def load_state_dict(path):
    ckpt = torch.load(path, map_location="cpu")

    # Case 1: saved via torch.save(model.state_dict())
    if isinstance(ckpt, dict) and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        return ckpt

    # Case 2: common training checkpoint formats
    if "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]

    if "state_dict" in ckpt:
        return ckpt["state_dict"]

    if "model" in ckpt:
        return ckpt["model"]

    raise ValueError("Could not find state_dict in checkpoint.")

def relative_diff(a, b, eps=1e-12):
    # Stable relative difference:
    # |a - b| / (|b| + eps)
    return torch.abs(a - b) / (torch.abs(b) + eps)

def compare_checkpoints(path1, path2):
    sd1 = load_state_dict(path1)
    sd2 = load_state_dict(path2)

    keys1 = set(sd1.keys())
    keys2 = set(sd2.keys())

    print("\n=== Key Comparison ===")
    print("Only in ckpt1:", keys1 - keys2)
    print("Only in ckpt2:", keys2 - keys1)

    shared_keys = sorted(keys1 & keys2)

    print(f"\nComparing {len(shared_keys)} shared layers...\n")

    results = []

    for k in shared_keys:
        w1 = sd1[k]
        w2 = sd2[k]

        if w1.shape != w2.shape:
            print(f"Shape mismatch for {k}: {w1.shape} vs {w2.shape}")
            continue

        diff = w1 - w2
        abs_diff = torch.abs(diff)

        max_abs = abs_diff.max().item()
        mean_abs = abs_diff.mean().item()
        l2_diff = torch.norm(diff).item()

        # Relative difference
        rel = relative_diff(w1, w2)
        mean_rel = rel.mean().item()
        max_rel = rel.max().item()

        # Cosine similarity (flatten first)
        cos = torch.nn.functional.cosine_similarity(
            w1.flatten(), w2.flatten(), dim=0
        ).item()

        results.append((k, max_abs, mean_abs, l2_diff, mean_rel, max_rel, cos))

    # Sort by mean absolute diff (largest change first)
    results.sort(key=lambda x: x[2], reverse=True)

    print("=== Layer-wise Differences ===")
    for r in results:
        print(f"\nLayer: {r[0]}")
        print(f"  max_abs_diff   : {r[1]:.6e}")
        print(f"  mean_abs_diff  : {r[2]:.6e}")
        print(f"  l2_diff        : {r[3]:.6e}")
        print(f"  mean_rel_diff  : {r[4]:.6e}")
        print(f"  max_rel_diff   : {r[5]:.6e}")
        print(f"  cosine_sim     : {r[6]:.6f}")

    print("\nDone.")

compare_checkpoints('mws_to_mwp/ckpt_after_MWS.pt', 'mws_to_mwp/ckpt_after_MWP.pt')


=== Key Comparison ===
Only in ckpt1: set()
Only in ckpt2: set()

Comparing 22 shared layers...

=== Layer-wise Differences ===

Layer: transformer.h.0.ln_2.weight
  max_abs_diff   : 1.814461e-02
  mean_abs_diff  : 5.566929e-03
  l2_diff        : 1.043933e-01
  mean_rel_diff  : 5.460938e-03
  max_rel_diff   : 1.759815e-02
  cosine_sim     : 0.999994

Layer: transformer.ln_f.weight
  max_abs_diff   : 1.114309e-02
  mean_abs_diff  : 4.759026e-03
  l2_diff        : 8.292861e-02
  mean_rel_diff  : 4.652448e-03
  max_rel_diff   : 1.085967e-02
  cosine_sim     : 0.999998

Layer: lm_head.weight
  max_abs_diff   : 1.773964e-02
  mean_abs_diff  : 4.221855e-03
  l2_diff        : 3.695652e-01
  mean_rel_diff  : 8.889530e-01
  max_rel_diff   : 2.586947e+02
  cosine_sim     : 0.985811

Layer: transformer.h.0.mlp.c_fc.weight
  max_abs_diff   : 2.070928e-02
  mean_abs_diff  : 3.596440e-03
  l2_diff        : 2.302642e+00
  mean_rel_diff  : 1.427601e+00
  max_rel_diff   : 1.018830e+04
  cosine_sim    